# L07 · 万物皆正弦 — 用三角波叠加合成方波，傅里叶直觉一图彻底建立

**目标**：很多正弦叠加能拼出复杂波形；反过来，任何波形都能拆成正弦成分。

**为什么对 Aurora 重要**：`aurora.audio.fft` 中的 FFT 和 STFT 做的正是这个拆解——把采样信号还原为各频率的幅度和相位，频谱图的每一列都是一次傅里叶分析的结果。

## 本课剧情：拆开复杂声音的合唱

复杂波形可以是多个正弦波同时叠加的结果。把叠加过程逆回来，就能读出每个频率的成分——这就是傅里叶分析的全部直觉。本节先动手叠加奇次谐波，观察 n 项之和如何逼近方波。

## 1. 叠加几个正弦 → 复杂波形

两个不同频率的正弦叠加，结果已经不是正弦——波形形状由各分量的频率比和权重共同决定。把频率取奇数倍基频、权重设为 `1/k`，叠加结果逐渐逼近方波。本质原因：方波是奇函数，傅里叶级数中只含奇次正弦项；系数 `1/k` 的衰减保证级数收敛，物理上对应谐波能量随频率升高而减弱。

## 实验入口：观察谐波叠加

每增加一个奇次谐波，合成波形的边缘就更陡一点，顶部就更平一点。这一格先画基础的 2 Hz + 5 Hz 叠加，后续格把它推广到任意项数。

In [ ]:
import numpy as np, matplotlib.pyplot as plt
t = np.linspace(0, 1, 500)
y = np.sin(2*np.pi*2*t) + 0.5*np.sin(2*np.pi*5*t)
plt.plot(t, y); plt.title('2 Hz + 0.5·(5 Hz)'); plt.show()

## 动手观察：每个分量长什么样

下面先把前几个奇次谐波单独打印出来。注意第 k 个分量的频率是基频的 k 倍，权重是 1/k——频率越高，贡献越小。

In [ ]:
import numpy as np

angles = np.array([0, np.pi/2, np.pi, 3*np.pi/2])
z = np.exp(1j * angles)

print('角度 =', np.round(angles, 3))
print('实部 cos =', np.round(z.real, 3))
print('虚部 sin =', np.round(z.imag, 3))
print('复数 z =', np.round(z, 3))


## 代码实验：叠加项数从 1 到 9

下面从 n=1 到 n=9 逐步增加谐波，观察合成波形的方波近似程度如何随项数提升。

In [ ]:
import numpy as np

for k in range(9):
    theta = 2 * np.pi * k / 8
    z = np.exp(1j * theta)
    print(f'k={k} theta={theta:.2f} -> ({z.real:+.3f}, {z.imag:+.3f})')


## 2. ✏️ 用奇次谐波叠加逼近方波 `square_approx(t, n)`

方波 ≈ `Σ_{k=1,3,5,...} sin(2π·k·t) / k`（取前 n 个奇次谐波）。

**推理路线**：
1. 输出需与 `t` 等长，先用 `np.zeros_like(t)` 初始化累加器 `result`，确保 dtype 和 shape 自动对齐。
2. 奇次谐波下标是 `1, 3, 5, ..., 2n-1`，用 `range(1, 2*n, 2)` 生成，共 `n` 项。
3. 循环内每次把 `np.sin(2*np.pi*k*t)/k` 加到 `result`——分母 `k` 正是让高频分量权重递减的关键，缺掉它级数不收敛。

**参考输入输出**：`n=1` 时只有基频，输出是标准正弦；`n=5` 时波形顶部已出现明显的"肩膀"；`n=50` 时方波的上下平台清晰可见，只在跳变处有吉布斯振铃。

<details><summary>点击查看参考实现</summary>

```python
def square_approx(t, n):
    result = np.zeros_like(t)
    for k in range(1, 2*n, 2):
        result += np.sin(2 * np.pi * k * t) / k
    return result
```

</details>

### 写代码前，先明确三件事

写 `square_approx` 前明确三件事：
- 输入：`t`（时间数组）、`n`（叠加的谐波个数）
- 关键步骤：对 `k in [1, 3, 5, ..., 2n-1]` 累加 `sin(2π·k·t) / k`
- 返回：与 `t` 等长的一维数组，值域约为 `[-π/4, π/4]`

In [ ]:
import numpy as np
def square_approx(t, n):
    out = np.zeros_like(t)
    # ✏️ TODO: 累加前 n 个奇次谐波
    return out


In [ ]:
import numpy as np, matplotlib.pyplot as plt
t = np.linspace(0, 1, 1000)
for n in [1, 3, 10]:
    plt.plot(t, square_approx(t, n), label=f'{n} harmonics')
plt.legend(); plt.title('harmonics build a square wave'); plt.show()
# 自检: 谐波越多, 越接近 ±π/4 的方波幅度
approx = square_approx(t, 50)
assert approx.max() > 0.7, '叠加足够多谐波后应接近方波'
print('✅ 通过：你亲手用正弦拼出了方波 = 傅里叶级数。')

**🔗 Aurora 连接**：傅里叶变换把信号拆回这些正弦成分(频谱)；Week 2 的 FFT、Week 3 的频谱图，做的就是这件事。

**🎉 数学前导全部完成**：线代·微积分·概率·复数三角四块地基齐了，足以支撑 Aurora 普通工程的核心数学。

In [ ]:
for k in range(8):
    theta = 2*np.pi*k/8
    z = np.exp(1j*theta)
    radius = abs(z)
    print(f'k={k} | z=({z.real:+.2f}, {z.imag:+.2f}) | 半径={radius:.2f}')
print('所有点半径都接近 1，说明它们都在单位圆上。')


## 参数实验：只改一个旋钮

把 `n` 从 1 增加到 20，观察方波顶部的平坦度如何提高。注意 `t=0.5` 附近的跳变处：即使 `n` 很大，该处的超调量始终接近 9%（吉布斯现象），增加 `n` 无法消除这个振铃，只能使其向跳变点收缩。

In [ ]:
for k in range(8):
    theta = 2*np.pi*k/8
    z = np.exp(1j*theta)
    radius = abs(z)
    print(f'k={k} | z=({z.real:+.2f}, {z.imag:+.2f}) | 半径={radius:.2f}')
print('所有点半径都接近 1，说明它们都在单位圆上。')


## 本课收束

现在能用 `square_approx(t, n)` 生成任意项数的谐波叠加，并观察合成波形随 n 收敛的过程。这直接对应 `aurora.audio.fft` 的工作原理：FFT 输出的每个频率桶是某个正弦分量的幅度和相位。STFT 在每个时间帧重复一次这个拆解，得到频谱图的一列。Week 1 实现离散 FFT 时，会用方波信号验证频谱中只有奇次谐波出现。

In [ ]:
# 小检查：同一个角度，同时看 cos、sin、复数
import numpy as np

for theta in [0, np.pi/2, np.pi, 3*np.pi/2, 2*np.pi]:
    z = np.exp(1j * theta)
    print(f'theta={theta:5.2f} -> cos={z.real:+.1f}, sin={z.imag:+.1f}, complex={z.real:+.1f}{z.imag:+.1f}j')
